# RL Project - Q-Learning Based Approach to Implement Dynamic Traffic Signals
The project works on a traffic signal agent that allows for responding to varying states of congestion.

In [34]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IntersectionEnv import IntersectionEnv
from QLearningAgent import QLearningAgent

In [35]:
    # --- Simulation Parameters ---
# Arrival rates (lambda) for the 8 lanes (vehicles per second)
# Assuming a uniform flow baseline for now
arrival_rates = [0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05] 
dequeue_rate = 0.5    # Vehicles cleared per second on green
change_penalty = 10.0 # Eta: Penalty for switching phases to prevent flickering

episodes = 5
steps_per_episode = 3600 # 1 hour of simulated time per episode

# --- Initialization ---
env = IntersectionEnv(arrival_rates, dequeue_rate, change_penalty)
agent = QLearningAgent(action_space_size=4)


In [10]:
%%prun -D profile_results.pstat
from tqdm.auto import tqdm

# Tracking metrics for evaluation
episode_rewards = []

print("Starting training phase...")

tepisodes = tqdm(range(episodes))

# --- Main Training Loop ---
for episode in tepisodes:
    tepisodes.set_description(desc=f'Episode {episode}')
    
    # Reset the environment at the start of each episode
    env.__init__(arrival_rates, dequeue_rate, change_penalty)
    state = env.get_discrete_state()
    total_reward = 0
    
    tsteps_per_episode = tqdm(range(steps_per_episode), leave=False)
    for step in tsteps_per_episode:
        tsteps_per_episode.set_description(desc=f'Step {step}')
        
        # 1. Agent chooses an action (phase)
        action = agent.choose_action(state)
        
        # 2. Environment steps forward based on the action
        next_state, reward = env.step(action)
        
        # 3. Agent learns from the consequences
        agent.learn(state, action, reward, next_state)
        
        # 4. Transition to the next state
        state = next_state
        total_reward += reward
        
    # Decay exploration rate at the end of the episode
    agent.decay_epsilon()
    episode_rewards.append(total_reward)
    
    # Print progress every 100 episodes
    if (episode + 1) % 100 == 0:
        avg_reward = np.mean(episode_rewards[-100:])
        print(f"Episode: {episode + 1:4d} | Epsilon: {agent.epsilon:.3f} | Avg Reward (Last 100): {avg_reward:.0f}")

print("Training complete!")

Starting training phase...


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/3600 [00:00<?, ?it/s]

  0%|          | 0/3600 [00:00<?, ?it/s]

  0%|          | 0/3600 [00:00<?, ?it/s]

  0%|          | 0/3600 [00:00<?, ?it/s]

  0%|          | 0/3600 [00:00<?, ?it/s]

Training complete!
 
*** Profile stats marshalled to file 'profile_results.pstat'.


         14957417 function calls (14083619 primitive calls) in 180.132 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    85345   96.002    0.001   98.407    0.001 IntersectionEnv.py:40(_process_arrivals)
170887/46622   21.464    0.000   20.560    0.000 socket.py:623(send)
768105/85345    9.789    0.000   10.178    0.000 {built-in method builtins.sum}
20324/3972    2.843    0.000    1.352    0.000 {built-in method select.select}
    19424    1.744    0.000    2.607    0.000 std.py:464(format_meter)
2518275/2477616    1.714    0.000    1.992    0.000 {built-in method builtins.isinstance}
    18000    1.650    0.000    1.650    0.000 QLearningAgent.py:24(choose_action)
    86284    1.553    0.000    2.758    0.000 encoder.py:205(iterencode)
   682760    1.451    0.000    1.964    0.000 numeric.py:1975(isscalar)
    40669    1.392    0.000    1.531    0.000 attrsettr.py:66(_get_attr_opt)
    18000    1.126    0.000  106.905  

In [7]:
import pstats
from pstats import SortKey

# 2. Load the saved profiling data
p = pstats.Stats('profile_results.pstat')

# 3. Sort by cumulative time and print functions along with their subfunctions
p.strip_dirs().sort_stats(SortKey.TIME).print_stats("Env", 10)
p.print_callers(.5)

Thu May 14 15:36:31 2026    profile_results.pstat

         14957417 function calls (14083619 primitive calls) in 180.132 seconds

   Ordered by: internal time
   List reduced from 474 to 8 due to restriction <'Env'>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    85345   96.002    0.001   98.407    0.001 IntersectionEnv.py:40(_process_arrivals)
    18000    1.126    0.000  106.905    0.006 IntersectionEnv.py:63(step)
   606105    0.571    0.000    7.031    0.000 IntersectionEnv.py:73(<genexpr>)
    18000    0.429    0.000    0.481    0.000 IntersectionEnv.py:53(_process_departures)
    18005    0.299    0.000    0.448    0.000 IntersectionEnv.py:23(get_discrete_state)
    18000    0.211    0.000    0.261    0.000 IntersectionEnv.py:92(_get_lanes_for_phase)
   162000    0.141    0.000    1.948    0.000 IntersectionEnv.py:85(<genexpr>)
        5    0.001    0.000    0.001    0.000 IntersectionEnv.py:6(__init__)


   Ordered by: internal time
   List reduced 

In [11]:
print(agent.q_table)

defaultdict(<function QLearningAgent.__init__.<locals>.<lambda> at 0x0000015F91D6FEC0>, {(0, 0, 0, 0, 0, 0, 0, 0, 0): array([  0.   , -10.36 , -41.554, -67.417]), (0, 0, 0, 0, 0, 0, 0, 0, 1): array([ -24.8  ,  -15.472, -109.07 ,    0.   ]), (0, 0, 0, 0, 0, 0, 0, 0, 2): array([-128.791, -105.4  ,   -2.1  ,    0.   ]), (0, 0, 0, 1, 0, 0, 0, 0, 0): array([   0. , -162.5,    0. ,    0. ]), (0, 0, 1, 1, 0, 0, 0, 0, 1): array([   0. ,    0. , -209. , -336.8]), (0, 0, 1, 1, 0, 0, 0, 0, 2): array([   0. , -267.8,    0. ,    0. ]), (0, 1, 1, 2, 0, 0, 0, 0, 3): array([-415.,    0.,    0.,    0.]), (0, 1, 1, 2, 0, 0, 0, 0, 0): array([   0. , -497.6,    0. ,    0. ]), (1, 1, 1, 2, 0, 0, 0, 0, 1): array([   0. ,  -91.4, -602. ,    0. ]), (1, 1, 1, 2, 0, 0, 0, 0, 2): array([   0. ,    0. ,    0. , -697.1]), (1, 1, 1, 2, 0, 0, 1, 1, 3): array([-826.6,    0. ,    0. , -126.1]), (1, 1, 2, 2, 1, 0, 1, 1, 0): array([ -296.4 , -2740.59, -1527.2 ,     0.  ]), (1, 1, 2, 2, 1, 0, 1, 1, 1): array([    0. ,   